# Sabeel: resume pothole training in a new session

Use this when an earlier session was cut off (disconnect, idle timeout, GPU limit).
**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

1. Settings and Drive  2. Install  3. Restore the dataset from Drive (~1 min)  4. Continue training from the last saved epoch
5–8. Test metrics, ONNX export, CPU speed, sample predictions (also run when training had already finished)

Run it as many times as needed: each run continues from where the last one stopped.

In [ ]:
# 1. Settings, GPU check, Drive
MODEL = "yolo11s.pt"  # "yolo11n.pt" = faster, a bit less accurate
EPOCHS = 100          # use 1 for a quick pipeline test
IMGSZ = 640           # keep 640: matches ml-training/inference.py and keeps CPU latency low
NEG_RATIO = 0.15      # pothole-free road photos, as a share of pothole photos
RDD_URL = "https://ndownloader.figshare.com/files/38030910"   # official RDD2022 on figshare (13.3 GB); the old S3 link returns 403

import subprocess
assert subprocess.run(["nvidia-smi"]).returncode == 0, "No GPU: Runtime → Change runtime type → T4 GPU"

from google.colab import drive
drive.mount("/content/drive")
SAVE = "/content/drive/MyDrive/sabeel"
!mkdir -p {SAVE}

In [ ]:
# 2. Install
!pip install -q -U ultralytics onnxruntime

In [ ]:
# 3. Restore the built dataset from Drive (saved there by the training notebook)
import os, glob
from pathlib import Path
CACHE = f"{SAVE}/data_cache.zip"
assert os.path.exists(CACHE), (
    "No dataset cache on Drive. Run cells 1-4 of sabeel_pothole_training.ipynb once (they rebuild and cache it), then come back here.")
!unzip -q -o "{CACHE}" -d /content
counts = {s: len(glob.glob(f"/content/data/{s}/images/*")) for s in ("train", "val", "test")}
print("restored dataset:", counts)
assert counts["train"] > 0 and os.path.exists("/content/data.yaml"), "cache is incomplete: delete it on Drive and rebuild with the training notebook"

In [ ]:
# 4. Continue training from the last saved epoch
import torch
from ultralytics import YOLO
LAST = f"{SAVE}/runs/pothole/weights/last.pt"
assert os.path.exists(LAST), f"No checkpoint at {LAST}: start training with cell 5 of sabeel_pothole_training.ipynb first"
ckpt = torch.load(LAST, map_location="cpu", weights_only=False)
# a finished run has its optimizer stripped and epoch set to -1; resuming it would make ultralytics start an unrelated default training
if ckpt.get("optimizer") is None or ckpt.get("epoch", -1) < 0:
    print("Training already finished, nothing to resume. Run the cells below for metrics and export.")
else:
    print(f"resuming at epoch {ckpt['epoch'] + 2} of {ckpt.get('train_args', {}).get('epochs', '?')}")
    YOLO(LAST).train(resume=True)

In [ ]:
# 5. Tune the confidence threshold on VAL (max F1), then score once on TEST
import json, numpy as np
best = YOLO(f"{SAVE}/runs/pothole/weights/best.pt")

v = best.val(data="/content/data.yaml", split="val", imgsz=IMGSZ, plots=False,
             project=f"{SAVE}/runs", name="val", exist_ok=True)
try:
    f1 = np.asarray(v.box.f1_curve).mean(0)
    BEST_CONF = round(float(np.asarray(v.box.px)[f1.argmax()]), 3)
    print("best confidence threshold on val:", BEST_CONF, "| F1:", round(float(f1.max()), 3))
except Exception as e:
    BEST_CONF = 0.25
    print("threshold tuning unavailable in this ultralytics version, using 0.25:", e)

m = best.val(data="/content/data.yaml", split="test", imgsz=IMGSZ,
             project=f"{SAVE}/runs", name="test", exist_ok=True)
try:
    tf1 = np.asarray(m.box.f1_curve).mean(0)
    test_f1 = round(float(tf1[int(np.abs(np.asarray(m.box.px) - BEST_CONF).argmin())]), 4)
except Exception:
    test_f1 = None
metrics = {
    "model": MODEL, "epochs": EPOCHS, "imgsz": IMGSZ,
    "test_images": len(glob.glob("/content/data/test/images/*")),
    "precision": round(float(m.box.mp), 4), "recall": round(float(m.box.mr), 4),
    "mAP50": round(float(m.box.map50), 4), "mAP50-95": round(float(m.box.map), 4),
    "conf_threshold": BEST_CONF, "test_F1_at_threshold": test_f1,
}
print(metrics)

In [ ]:
# 6. Export to ONNX (fixed 640x640, same contract as ml-training/inference.py)
onnx_path = best.export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True, dynamic=False)
print("ONNX:", onnx_path)

In [ ]:
# 7. CPU latency of the ONNX model (Colab has 2 slow vCPUs, so our server will be at least this fast)
import time, onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0]
x = np.random.rand(1, 3, IMGSZ, IMGSZ).astype(np.float32)
for _ in range(3): sess.run(None, {inp.name: x})
t = []
for _ in range(20):
    t0 = time.perf_counter(); sess.run(None, {inp.name: x}); t.append((time.perf_counter() - t0) * 1000)
metrics["cpu_ms_colab"] = round(float(np.median(t)), 1)
metrics["onnx_output_shape"] = [int(s) for s in sess.run(None, {inp.name: x})[0].shape]   # expect [1, 5, 8400]
print(f"median CPU inference: {metrics['cpu_ms_colab']} ms | output {metrics['onnx_output_shape']}")
Path(f"{SAVE}/test_metrics.json").write_text(json.dumps(metrics, indent=2))
!ls -la {SAVE}/runs/pothole/weights {SAVE}

In [ ]:
# 8. Visual check: predictions on a few test images at the tuned threshold
from IPython.display import Image as Show, display
for p in sorted(glob.glob("/content/data/test/images/*.jpg"))[:8]:
    best.predict(p, conf=BEST_CONF, imgsz=IMGSZ, verbose=False)[0].save(filename="/content/pred.jpg")
    display(Show("/content/pred.jpg", width=480))

## Then
From Google Drive → `MyDrive/sabeel`, download `runs/pothole/weights/best.onnx` and `test_metrics.json`.